In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import gcamreader
import pandas as pd

def string_to_xml_file(xml_string, file_name):
    """
    Converts a string into a well-formatted (indented) XML file, without unnecessary newlines.

    Parameters:
    xml_string (str): The XML content as a string.
    file_name (str): The desired filename for the XML file.
    """
    try:
        # Parse the XML string
        root = ET.ElementTree(ET.fromstring(xml_string))
        
        # Convert ElementTree to a string
        rough_string = ET.tostring(root.getroot(), encoding="utf-8")
        
        # Use minidom to pretty-print the XML
        parsed = minidom.parseString(rough_string)
        pretty_xml_as_string = parsed.toprettyxml(indent="  ")
        
        # Remove unnecessary blank lines created by toprettyxml()
        pretty_xml_as_string = "\n".join([line for line in pretty_xml_as_string.splitlines() if line.strip()])
        
        # Write the formatted XML to a file
        with open(file_name, "w", encoding="utf-8") as f:
            f.write(pretty_xml_as_string)
        
        print(f"XML file '{file_name}' created successfully with proper indentation and no extra newlines.")
    except ET.ParseError as e:
        print("Error parsing XML string:", e)

# Source

## Data Sources
- [KNREC](https://www.knrec.or.kr/biz/introduce/new_fin/intro_fin.do?gubun=A)

## Implemented Input Files
- `/input/policy/korea-2035/buildings/finance_support_cp.xml`  
- `/input/policy/korea-2035/buildings/finance_support_ep.xml`

# Financing Support for Renewable Energies in Commercial Buildings

The Ministry of Trade, Industry and Energy (MOTIE) provides long-term low-interest financing for individuals or companies seeking to install and operate new and renewable energy facilities, as well as for manufacturers of such facilities ([KNREC](https://www.knrec.or.kr/biz/introduce/new_fin/intro_fin.do?gubun=A)).  
As of Q4 2024, the interest rate is 1.75%. Assuming a normal corporate financing rate of 5.5%, this implies an effective interest subsidy of 3.75% for commercial heat-by-electricity technologies.  

In our assumptions, 35% of total financing demand is met under the *Current Policy* scenario, and 55% under the *Enhanced Policy* scenario.  
The subsidy amounts are calculated in line with the default technology cost assumptions in the model. Detailed implementation procedures are described below.

In [12]:
proj_path = Path("../../")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [13]:
xml_file_path = xml_path / "building_det.xml"
tree = ET.parse(xml_file_path)
root = tree.getroot()  # Get the root element of the XML
global_technology_database = root.find(".//global-technology-database")

In [14]:
lstSectorNm = []
lstYear = []
lstInputCost = []
for location_info in global_technology_database.findall(".//location-info[@subsector-name='electricity']"):
    sectorNm = location_info.get('sector-name')
    if "resid" in sectorNm:
        continue
    technology = location_info.find(".//technology[@name='electricity']")
    for period in technology.findall("period"):
        year = period.get('year')
        try:
            input_cost_val = period.find(".//input-cost").text
            lstSectorNm.append(sectorNm)
            lstYear.append(year)
            lstInputCost.append(input_cost_val)
        except:
            print(f"No input-cost found for {year} {sectorNm}")

In [15]:
dfInputCost = pd.DataFrame({'supplysector': lstSectorNm, 'year': lstYear, 'input-cost': lstInputCost})
dfInputCost = dfInputCost.drop_duplicates(['supplysector', 'input-cost']).drop(columns="year")

In [16]:
dfSubsidy = dfInputCost.copy()
dfSubsidy['input-cost'] = dfSubsidy['input-cost'].astype(float) * 0.0375 * 0.35
serSubsidy = dfSubsidy.set_index('supplysector')['input-cost']
serSubsidy.head()

supplysector
comm cooling    0.034519
comm heating    0.016799
comm others     0.034406
Name: input-cost, dtype: float64

In [18]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for suppplysectorNm in serSubsidy.index:
    supplysector = ET.SubElement(korea, 'supplysector', {'name': suppplysectorNm})
    subsector = ET.SubElement(supplysector, 'subsector', {'name': 'electricity'})
    stub_technology = ET.SubElement(subsector, 'stub-technology', {'name': 'electricity'})
    
    for year in range(2025, 2036, 5):
        period = ET.SubElement(stub_technology, 'period', {'year': str(year)})
        minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': 'heat-pump-financing'})
        input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
        input_cost.text = f"{-serSubsidy[suppplysectorNm]:.3f}"
        # input_cost.text = str(-serSubsidy[suppplysectorNm])

In [ ]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "finance_support_cp.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/finance_support_cp.xml' created successfully with proper indentation and no extra newlines.


In [19]:
dfSubsidyEp = dfInputCost.copy()
dfSubsidyEp['input-cost'] = dfSubsidyEp['input-cost'].astype(float) * 0.0375 * 0.55
serSubsidyEp = dfSubsidyEp.set_index('supplysector')['input-cost']
serSubsidyEp.head()

supplysector
comm cooling    0.054244
comm heating    0.026398
comm others     0.054066
Name: input-cost, dtype: float64

In [20]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for suppplysectorNm in serSubsidyEp.index:
    supplysector = ET.SubElement(korea, 'supplysector', {'name': suppplysectorNm})
    subsector = ET.SubElement(supplysector, 'subsector', {'name': 'electricity'})
    stub_technology = ET.SubElement(subsector, 'stub-technology', {'name': 'electricity'})
    
    for year in range(2025, 2036, 5):
        period = ET.SubElement(stub_technology, 'period', {'year': str(year)})
        minicam_non_energy_input = ET.SubElement(period, 'minicam-non-energy-input', {'name': 'heat-pump-financing'})
        input_cost = ET.SubElement(minicam_non_energy_input, 'input-cost')
        if year == 2025:
            input_cost.text = f"{-serSubsidy[suppplysectorNm]:.3f}"
        else:
            input_cost.text = f"{-serSubsidyEp[suppplysectorNm]:.3f}"

In [34]:
outfile_path = proj_path / "input" / "policy" / "ndc" / "buildings" / "finance_support_ep.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/data/project/tae/gcam-core/input/policy/ndc/buildings/finance_support_ep.xml' created successfully with proper indentation and no extra newlines.
